Data Analyst: Andrei Berner Arroyo Tanta

**Project 3 with Numpy: Monte Carlo Risk Engine for Financial Derivatives

**Business Application
In volatile market regimes, financial institutions must quantify the likelihood of extreme downside scenarios before committing capital. This project provides a quantitative framework for assessing "tail risk".

-Business Need: To simulate thousands of potential price trajectories for a large-cap stock (e.g., Meta) over a long-term horizon.

-Impacted Area: Risk Management, Treasury, and Quant Trading desks.

-Decision Support: Helps stakeholders set evidence-based drawdown limits and determine the "Probability of Ruin" for specific assets.

**Executive Summary

-Using vectorized path generation, the system simulated 10,000 trajectories over a 1,000-day horizon.

-Key Findings: The simulation revealed a neutral long-term drift (near-zero expected growth), but highlighted significant uncertainty in the final price distribution.

-Relevant Metrics: * Probability of Ruin: 0.42% (based on 42 breach events below the $570 threshold).

-Expected Growth: -0.0088%.

-Historical Extremes: A worst-case scenario price of $532.39 (-19.34%) and a best-case of $804.98 (+21.97%).

-Detected Risks: While the probability of insolvency remains low, the 42 breach events emphasize the need for monitoring extreme downside exposure during prolonged volatility.

In [1]:
#Simulation Architecture & Configuration
import numpy as np

In [2]:
#Simulation Parameters (Base: Meta Platforms Inc. @ $660)
initial_price = 660.0
mean_return = 0.0
volatility = 1.0  # Standard Deviation
time_horizon = 1000 # Trading Days
num_scenarios = 10000
ruin_threshold = 570.0 # ~13.6% drawdown limit

#Ensure Reproducibility
np.random.seed(42)

In [3]:
#Vectorized Path Generation
#Generate Daily Shocks: Normal Distribution (ufuncs)
#We generate 999 shocks to add to the initial starting price
daily_shocks = np.random.normal(mean_return, volatility, size=(num_scenarios, time_horizon - 1))

#Vectorized Integration: Cumulative Sum of shocks
cumulative_returns = np.cumsum(daily_shocks, axis=1)

#Construct Final Price Matrix: Memory-efficient Stacking
#Pre-allocating the starting price for all 10k paths
start_prices = np.full((num_scenarios, 1), initial_price)
price_paths = np.hstack([start_prices, initial_price + cumulative_returns])

print(f"Simulation Matrix Shape: {price_paths.shape}") # (10000 scenarios, 1000 days)

Simulation Matrix Shape: (10000, 1000)


In [4]:
#Risk Assessment: Probability of Ruin

#Identification of "Insolvency" events
#Any(axis=1) performs a vectorized search across each path
ruin_mask = (price_paths <= ruin_threshold).any(axis=1)
ruin_indices = np.where(ruin_mask)[0]

#Calculate Probability of Ruin
ruin_count = ruin_indices.size
prob_of_ruin = (ruin_count / num_scenarios) * 100

print(f"--- Report ---")
print(f"Threshold Breach Count: {ruin_count}")
print(f"Probability of Ruin: {prob_of_ruin:.2f}%")

--- Report ---
Threshold Breach Count: 42
Probability of Ruin: 0.42%


In [5]:
#Statistical Distribution Analysis
#Extracting Terminal Prices (Slicing the last column)
terminal_prices = price_paths[:, -1]

#Statistical Aggregations (Vectorized)
avg_terminal_price = np.mean(terminal_prices)
std_dev_terminal   = np.std(terminal_prices)
variance_terminal  = np.var(terminal_prices)

#Logic Check: Growth Expectancy
#Since mean shock = 0, the expected growth should be close to 0%
growth_pct = ((avg_terminal_price - initial_price) / initial_price) * 100

print(f"--- Report ---")
print(f"Average Final Price: ${avg_terminal_price:.2f}")
print(f"Standard Deviation: {std_dev_terminal:.2f}")
print(f"Expected Growth: {growth_pct:.4f}%")

--- Report ---
Average Final Price: $659.94
Standard Deviation: 31.17
Expected Growth: -0.0088%


In [6]:
#Scenario Stress Testing (Fancy Indexing)
##Global Extremes
abs_min = np.min(price_paths)
abs_max = np.max(price_paths)

#Identifying specific outlier paths
#Using np.where to locate the indices of the highest/lowest price reached
max_row, max_col = np.where(price_paths == abs_max)
min_row, min_col = np.where(price_paths == abs_min)

print(f"--- Report ---")
print(f"Worst Historical Price: ${abs_min:.2f} ({((abs_min-initial_price)/initial_price*100):.2f}%)")
print(f"Best Historical Price:  ${abs_max:.2f} (+{((abs_max-initial_price)/initial_price*100):.2f}%)")

--- Report ---
Worst Historical Price: $532.39 (-19.34%)
Best Historical Price:  $804.98 (+21.97%)


To evaluate long‑term downside exposure, the system simulated 10,000 potential price trajectories for a large‑cap stock (e.g., Meta) over a 1,000‑day horizon. This project helps quantify tail risk, estimate the probability of ruin, and understand the distribution of final outcomes under stochastic price dynamics.

Although the probability remains low (0.42%), the presence of 42 breach events highlights the importance of monitoring extreme downside scenarios, especially under prolonged volatility regimes.

The near‑zero expected growth rate indicates a neutral long‑term drift—derived from the volatility assumptions—while the moderate dispersion reflects meaningful uncertainty in the final price distribution.

Finally, the extreme gap between the worst and best historical prices illustrates the potential range of outcomes under both adverse and favorable market conditions, reinforcing the value of scenario‑based risk modeling.